# CSP Problem - Gruppe C

```Christopher Klein & Colin Vavra```

Für die Prüfungswochen der Hochschule werden **zwei große Prüfungsräume** benötigt. 

**Mehrere Studiengänge** greifen in denselben Wochen auf diese Räume zu. Die verfügbaren **Raum-Zeit-Slots** sollen so verteilt werden, dass die **unterschiedlichen Bedarfe** der Studiengänge berücksichtigt werden und gleichzeitig organisatorisch sinnvolle sowie **faire Belegungspläne** entstehen.

# Variablen + Wertebereiche festlegen

| Slot | Zeitraum    |
| ---- | ----------- |
| A    | 08:00–10:00 |
| B    | 10:30–12:30 |
| C    | 13:00–15:00 |
| D    | 15:30–17:30 |

Zwei Räume

Fünf Tage die Woche

Slots = 5 * 4 * 2 = 40

Studiengaenge = { WIW, ES, ET, INF, MB, MT, DS }

Die Studiengänge **unterscheiden sich in ihrer Größe**. Diese wird durch die **Anzahl der Kursgruppen**
beschrieben, die **in der jeweiligen Prüfungswoche** berücksichtigt werden müssen. Ein Studiengang
mit vielen Kursgruppen besitzt somit einen **höheren Bedarf an Prüfungsraumkapazität** als ein
Studiengang mit wenigen Kursgruppen.

Nicht jeder Studiengang ist in jeder Prüfungswoche beteiligt. Für jede Woche ist im Datensatz angegeben, welche Studiengänge in dieser Woche Prüfungen durchführen. Aus der *Anzahl der Kursgruppen* dieser Studiengänge ergibt sich der **relative Bedarf innerhalb der jeweiligen Woche**.

=> Ein CSP pro Woche

# Daten

## exam_weeks.csv
- Woche 1-3
- Studiengang
- Anzahl Gruppen

## preferences.json
- Time Slot Preference (1-5)
- Weekday Preference (1-5)
- Structure Preference
  - 3 Preferences
  - Jede enthält
    - condition gegen Anzahl course_groups
    - Parallel vs Sequentiell Preference
    - Preferenzwert + allowed-Boolean

# Wochenplan lesen

In [613]:
import pandas as pd

exam_weeks = pd.read_csv('./data/exam_weeks.csv', delimiter=';')
exam_weeks.columns = ['exam_week','study_program','course_groups','cohorts']
#exam_weeks.head()
print(exam_weeks.to_string())

   exam_week study_program  course_groups  cohorts
0    Woche 1            ES              3        1
1    Woche 1           INF              1        1
2    Woche 1            MB              8        1
3    Woche 1            MT              4        1
4    Woche 1            DS              4        1
5    Woche 2            ET              2        1
6    Woche 2            MB              8        1
7    Woche 2            MT              4        1
8    Woche 2            DS              4        1
9    Woche 3           WIW              5        1
10   Woche 3            ET              2        1
11   Woche 3            MB              8        1
12   Woche 3            DS              4        1


# Präferenzen lesen

In [614]:
import json

with open('./data/preferences.json') as f:
    prefs = json.load(f)

# Popularität der Zeitslots (A-D) und Wochentage
print("Zeitslots:", {t: v["preference"] for t, v in prefs["time_slot_preferences"].items()})
print("Wochentage:", prefs["weekday_preferences"])

# Regeln für parallele / aufeinanderfolgende Termine
print("\nStruktur-Regeln:")
for rule in prefs["appointment_structure_preferences"]:
    print(f'  {rule["condition"]}; parallel: {rule["parallel"]},  aufeinanderfolgend: {rule["sequential"]}')

Zeitslots: {'A': 4, 'B': 5, 'C': 2, 'D': 2}
Wochentage: {'Montag': 5, 'Dienstag': 4, 'Mittwoch': 4, 'Donnerstag': 3, 'Freitag': 1}

Struktur-Regeln:
  course_groups >= 3; parallel: {'preference': 6, 'allowed': True},  aufeinanderfolgend: {'preference': 3, 'allowed': True}
  course_groups == 2; parallel: {'preference': 0, 'allowed': True},  aufeinanderfolgend: {'preference': 0, 'allowed': True}
  course_groups == 1; parallel: {'preference': 0, 'allowed': False},  aufeinanderfolgend: {'preference': 0, 'allowed': False}


# Vorüberlegungen

CSP-basiertes Planungssystem, das für die im Datensatz beschriebenen Prüfungswochen *jeweils* eine *zulässige Verteilung* der *verfügbaren Raum-Zeit-Slots* auf die beteiligten Studiengänge bestimmt.

- Verteilung pro Woche 
- Slot zu Studiengang

```Eine mögliche Modellierung besteht beispielsweise darin, jeder Kombination aus Wochentag, Zeitslot und Raum eine Variable zuzuordnen, deren Wert den zugeordneten Studiengang angibt. Andere fachlich geeignete Repräsentationen sind zulässig.```

## 4.1 - Modell

### Variablen

X_1, X_2, ..., X_n 

Tag + Slot + Raum


### Domains

Menge der Kurse (study_program)

D(x) = study_program

Beispiel: ``` D("Mo_A_R1") = "MB"```

### Fragen aus Aufgabe zu 4.1

- Welche Entscheidungen muss das Planungssystem treffen?
  - *zulässige Verteilung* der *verfügbaren Raum-Zeit-Slots* auf die beteiligten Studiengänge
  - es wird seitens Prof ausdrücklich keine Findung einer Optimallösung gefordert
  - entsprechend genügt "zulässige" Lösung
  - CSP findet bei mehreren Läufen unterschiedliche Lösungen

- Wie lässt sich aus der Zahl der Kursgruppen der relative Bedarf ableiten?
  - siehe 4.2

- Welche organisatorischen Anforderungen werden als Constraints formuliert?
  - **Bedarf.** Anteilige Bedarfsermittlung eines Studienganges einmal pro Woche und einmal pro Tag.
  - **Fairness.** Berücksichtigung der Popularität einzelner Tage und Slots.
  - **Struktur.** Berücksichtigung von Parallelisierung und Sequentialisierung von Studiengängen abhängig von Präferenz.
  - weitere aus 4.5 (siehe unten)

- Welche Bedingungen betreffen einzelne Raum-Zeit-Slots und welche koppeln mehrere Zuordnungen miteinander?
  - Sämtliche o.g. Constraints bedürfen einer Betrachtung mehrerer Slots in Bezug aufeinander



# Init

In [615]:
import math

# Mengen definieren
weekdays = ["Mo", "Di", "Mi", "Do", "Fr"]
slots = ["A", "B", "C", "D"]
rooms = ["R1", "R2"]

# X: eine Variable für jede Kombination aus Tag, Zeitslot und Raum
variables = [f"{day}_{slot}_{room}" for day in weekdays for slot in slots for room in rooms]
N_SLOTS = len(variables)  # 40
SLOTS_PER_DAY = len(slots) * len(rooms)  # 4 Slots in jeweils 2 Räumen = 8 Slots

weeks = exam_weeks["exam_week"].unique().tolist()
print(weeks, f", Slots: {N_SLOTS}", f", Slots per Day: {SLOTS_PER_DAY}")

['Woche 1', 'Woche 2', 'Woche 3'] , Slots: 40 , Slots per Day: 8


# CSP System

Umsetzung mit OR-Tools (CP-SAT)

- Variable = Tag + Slot + Raum
- Wert = Studiengang

Darstellung der Variablen durch Boolesche Variablen $x_{v,s} \in \{0,1\}$: 

$x_{v,s} = 1$ heißt „Slot $v$ gehört Studiengang $s$“.

Constraints werden nach dem folgenden Block erklärt.

In [ ]:
# CSP System

from ortools.sat.python import cp_model

def build_csp(week):
    week_df = exam_weeks[exam_weeks["exam_week"] == week]
    programs = week_df["study_program"].tolist() # nur Studiengänge dieser Woche (--> quasi Domain!)

    model = cp_model.CpModel()
    x = {
        (v, s): model.NewBoolVar(f"{v}_{s}") 
        for v in variables # 'variables' kommt von der Definition am Anfang des Dokuments 
        for s in programs 
    } 
    # Beispiel: ('Mo_A_R1', 'ES'): Mo_A_R1_ES(0..1)

    # In einem Slot kann nur genau 1 Studiengang enthalten sein (umgkehrt bereits durch Modell abgesichert)
    for v in variables:
        model.AddExactlyOne(
            x[v, s] for s in programs
        )

    # Bedarfe (4.2)

    # -> Bedarf pro Woche
    week_demand = demand_bounds(week_df)    
    for s, (lo, hi) in week_demand.items():
        model.AddLinearConstraint(
            sum(x[v, s] for v in variables), # Ergibt lineare Summe: x_{1,s} + x_{2,s} + ... +  x_{n,s} -> wird erst später vom Solver ausgewertet !!
            lo, # Constraint garantiert Einhaltung der Untergrenze
            hi  # ... und Obergrenze
        )
    # => Solver weiß dann, dass Lösung für Lineare Gleichung zwischen LO und HI liegen muss        

    # -> Bedarf pro Tag (analog Woche)
    for day in weekdays:
        day_vars = [v for v in variables if v.startswith(day)] # nur Variablen des betroffenen Tages heranziehen
        day_demand = demand_bounds(week_df, SLOTS_PER_DAY)

        for s, (lo, hi) in day_demand.items():
            model.AddLinearConstraint(
                sum(x[v, s] for v in day_vars), 
                lo, 
                hi
            )

    # Popularität (4.3)
    add_popularity_constraints(model, x, programs) # siehe unten
    
    # Parallele / Aufeinanderfolgende Termine (4.4)
    add_structure_constraints(model, x, week_df, programs) # siehe unten

    # Kein Übergang spät -> früh (4.5)
    add_transition_constraints(model, x, programs) # siehe unten
    

    return model, x


def solve(model, x):
    solver = cp_model.CpSolver()
    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None
    return {v: s for (v, s), var in x.items() if solver.Value(var)}

In [617]:
def show_plan(solution):
    rows = [
        {
            "Tag": v.split("_")[0], 
            "Slot": v.split("_")[1], 
            "Raum": v.split("_")[2], 
            "SG": s # study_program
        }
        for v, s in solution.items()
    ]
    
    #plan = pd.DataFrame(rows)
    #print (plan)
    #     Tag   Slot Raum  SG
    # 0   Mo    A    R1    INF
    # 1   Mo    A    R2    DS

    plan = pd.DataFrame(rows).pivot(
        index=["Slot", "Raum"], 
        columns="Tag", 
        values="SG"
    )
    return plan[weekdays]

def slots_per_day(solution):
    rows = [
        {
            "Tag": v.split("_")[0], 
            "SG": s # study_program
        } 
        for v, s in solution.items()
    ]

    df = pd.DataFrame(rows)
    return pd.crosstab(
        df["SG"], 
        df["Tag"]
    )[weekdays]

solutions = {}
for week in weeks:
    print("========================================================")
    print(f"Week: {week}")
    print("========================================================")

    solutions[week] = solve(*build_csp(week))
    # e.g. {'Mo_A_R1': 'INF', 'Mo_A_R2': 'DS', 'Mo_B_R1': 'MB', ........ }

    print("Week plan:")
    display(show_plan(solutions[week]))
    print("")

    print("Week demand:")
    print(week, pd.Series(solutions[week]).value_counts().to_dict()) # print how often each course de-facto occurs in this week (cmp. demand_bounds())
    print("")

    print("Day demand:")
    display(slots_per_day(solutions[week]))
    print("")

    print("Popularität und Struktur:")
    display(pop_and_struc(solutions[week]))

Week: Woche 1
Week plan:


Tag         Mo  Di   Mi  Do  Fr
Slot Raum                      
A    R1     MT  MB   DS  DS  MB
     R2     MT  DS   DS  DS  MB
B    R1     MB  MB   MB  MT  ES
     R2     MB  MT   ES  MT  DS
C    R1     MB  ES   MT  MB  MT
     R2     ES  ES  INF  MB  DS
D    R1    INF  MB   MB  ES  MB
     R2     DS  MB   MB  MB  MT


Week demand:
Woche 1 {'MB': 16, 'MT': 8, 'DS': 8, 'ES': 6, 'INF': 2}

Day demand:


Tag,Mo,Di,Mi,Do,Fr
SG,,,,,
DS,1,1,2,2,2
ES,1,2,1,1,1
INF,1,0,1,0,0
MB,3,4,3,3,3
MT,2,1,1,2,2



Popularität und Struktur:


...

Tag         Mo   Di   Mi   Do   Fr
Slot Raum                         
A    R1     MB   MB  WIW  WIW  WIW
     R2     MB   MB   MB   ET   MB
B    R1    WIW   DS   DS   MB   ET
     R2    WIW  WIW   MB   DS   MB
C    R1     ET   MB   ET  WIW   DS
     R2     MB   MB   DS   DS   DS
D    R1     DS  WIW  WIW   MB   MB
     R2     DS  WIW   MB   MB  WIW


Week demand:
Woche 3 {'MB': 16, 'WIW': 11, 'DS': 9, 'ET': 4}

Day demand:


Tag,Mo,Di,Mi,Do,Fr
SG,,,,,
DS,2,1,2,2,2
ET,1,0,1,1,1
MB,3,4,3,3,3
WIW,2,3,2,2,2



Popularität und Struktur:


,Slots,mittl. Popularität,parallel,aufeinanderfolgend
SG,,,,
DS,9,6.33,2,2
ET,4,6.50,0,0
MB,16,6.75,4,2
WIW,11,6.82,2,0


# 4.1 Modell

- X = Tag + Slot + Raum
- D = { MB, INF, ...} // 5
- C = Constraints

Constraints:
- Nur Studiengänge der Woche ist durch dedizierte CSP's pro Woche gegeben
- Jeder Slot gehört genau einem Studiengang.
    - Lösung durch `AddExactlyOne`
    - **Anders gesagt:** Summe der Boolean-Werte eines Studiengangs innerhalb eines Slots **ist genau 1**.


# 4.2 Bedarf pro Woche und pro Tag

Es gilt:

- $g_s$: Anzahl der Kursgruppen des Studiengangs $s$
- $n_{\text{slots}}$: Anzahl verfügbarer Slots, z. B. 40 pro Woche oder 8 pro Tag
- $N_{\text{Woche}}$: Gesamtzahl der Kursgruppen in dieser Woche
- $n_s$: Anzahl der Slots, die dem Studiengang zugeordnet werden

#### Woche 

Anteil der Kursgruppen des Studiengangs in dieser Woche:
$$
q_s = \frac{g_s}{N_{\text{Woche}}}
$$

Anteilig gegen die Anzahl der Wochen-Slots rechnen:
$$bounds = 40\,q_s$$

Abrunden für Untergrenze, Aufrunden für Obergrenze.

Tatsächlicher Bedarf muss zwischen Untergrenze und Obergrenze liegen:
$$\lfloor bounds \rfloor \le n_s \le \lceil bounds \rceil$$

Beispiel Woche 1: 
- MB 16
- ES 6
- MT 8
- DS 8
- INF 2 

Slots für die gesamte Woche. 

Rechenweg als Beispiel für "MB" dazu:
$$
q_s = 8/20 = 0,4
$$
$$
bounds = 40 * 0,4 = 16
$$

Da ganzzahlig ergibt sich sowohl für Untergrenze als auch Obergrenze der Wert **16**.


#### Tag 
Damit sich die Slots eines Studiengangs nicht auf einzelne Tage konzentrieren, gilt dieselbe Regel für die 8 Raum-Zeit-Slots jedes Tages (4 Zeitslots × 2 Räume):
$$\lfloor 8\,q_s \rfloor \le n_{s,d} \le \lceil 8\,q_s \rceil \quad \text{für jeden Tag } d$$

Der tägliche Bedarf ergibt sich damit, wie gefordert, aus der Größe des Studiengangs **und** aus den anderen Studiengängen derselben Woche, weil beide in $q_s$ eingehen. 

Beispiel Woche 1: 
- MB 3–4
- ES 1–2
- MT 1–2
- DS 1–2
- INF 0–1 

Slots pro Tag. 

Ist $\lfloor 8\,q_s \rfloor \ge 1$, ist der Studiengang automatisch an jedem Tag vertreten (vgl. 4.5).

`demand_bounds(week_df, n_slots)` berechnet beide Grenzen, für die Woche mit `n_slots=40`, für den Tag mit `n_slots=8`.

In [618]:
def demand_bounds(week_df, n_slots=N_SLOTS):

    # Studiengänge zu Kursgruppen
    groups = dict(zip(
        week_df["study_program"],
        week_df["course_groups"] * week_df["cohorts"]
    ))

    total_of_week = sum(groups.values()) # Gesamtbedarf aller Gruppen dieser Woche

    # Anteiliger Bedarf
    results = {}  
    for s, g in groups.items(): # s = StudyProgram, g=Kursgruppen
        q_s = g / total_of_week # Anteil Gruppen des Studiengangs an allen Gruppen der Woche
        bound = q_s * n_slots # Mit diesem Quotienten die Slots anteilig ermitteln aus Gesamtslots (Woche = 40, Tag = 8)       

        results[s] = (
            math.floor(bound),  # lo
            math.ceil(bound)   # hi
        )        

    return results
    # ergibt Return Wert mit Dict s: (lo, hi)



In [619]:
week_df = exam_weeks[exam_weeks["exam_week"] == "Woche 1"]
print (week_df)

bounds_week = demand_bounds(week_df)
print(f"bounds_week: {bounds_week}")

bounds_day = demand_bounds(week_df, 8)
print(f"bounds_day: {bounds_day}")

  exam_week study_program  course_groups  cohorts
0   Woche 1            ES              3        1
1   Woche 1           INF              1        1
2   Woche 1            MB              8        1
3   Woche 1            MT              4        1
4   Woche 1            DS              4        1
bounds_week: {'ES': (6, 6), 'INF': (2, 2), 'MB': (16, 16), 'MT': (8, 8), 'DS': (8, 8)}
bounds_day: {'ES': (1, 2), 'INF': (0, 1), 'MB': (3, 4), 'MT': (1, 2), 'DS': (1, 2)}


# 4.3 Fairness bezüglich der Popularität

#### Popularitätsmaß
Tages- und Slotwert aus `preferences.json` werden addiert:
$$p(d,t) = \text{pref}_{Tag}(d) + \text{pref}_{Slot}(t)$$

(gemäß Aufgabenblatt, 4.3: ```Beispielsweise können die Werte von Tag und Zeitslot additiv kombiniert werden```)

Beispiel: 
- Montag A = *5 + 4 = 9*
- Freitag D = *1 + 2 = 3*

Popularität in Woche 1 über alle 40 Slots = $266$

Damit ergibt sich die mittlere Popularität $$\bar p = 266 / 40 = 6{,}65$$

#### Fairness-Constraint

Man kann nun für die **Popularität der von einem Studiengang verwendeten Slots aufsummieren** und **durch die vom Studiengang belegten Slots teilen** und erhält so die gemittelte Popularität bezogen auf die Slots dieses Studiengangs.

$$\textbf{Mittlere\_Popularität\_der\_Studiengang\_Slots} = \frac{\sum_{v} p(v) * \,x_{v,s}}{n_s}$$

Dieser Wert muss sich um den ermittelten Wert von $6,65$ befinden.

Das Intervall muss allerdings weiter nach links und rechts geöffnet werden. Hierzu wird der Toleranzwert $\tau$ verwendet.

$$6,65 - \tau \;\le\; \textbf{Mittlere\_Popularität\_der\_Studiengang\_Slots} \;\le\; 6,65 + \tau$$

**Achtung!** CP-SAT kann nicht mit Brüchen rechnen. Deshalb wird die Ungleichung umgeformt:
$$\left|\sum_v (40\,p(v) - 266)\,x_{v,s}\right| \le 40\,\tau\, n_s$$

Umformung gelang durch:
- **Nenner entfernen**: Multiplizieren mit Nenner n_s
- **Zusammenfassen**: Subtrahiere pMean (6,65) + Betrag
- **Dezimalzahl beseitigen**: pMean = 266/40 = 6,65 -> Einsetzen. 
  - Dann Ungleichung mit 40 multiplizieren
  - In Summe reinziehen


#### Wahl der Toleranz $\tau = 0{,}5$
Wahl von Toleranz führt teils zu nicht lösbarem Solving.....
| $\tau$ | Woche 1 | Woche 2 | Woche 3 |
|---|---|---|---|
| 0 | unlösbar | unlösbar | unlösbar |
| 0,1 | unlösbar | lösbar | lösbar |
| 0,25 | lösbar | lösbar | lösbar |
| 0,5 | lösbar | lösbar | lösbar |

Woche 1 scheitert bei $\tau = 0{,}1$ am Studiengang "INF":
- Beim Teilen durch 1 oder 2 kommt entweder wieder Ganzzahl oder .5 heraus
- 6,65 +/- 0,1 führt hier zu ungültigem Intervall mit 6,55 bzw. 6,75 
- Solver kann keine Lösung finden!

In [620]:
# Wir erstellen ein Mapping:
# Wochentage im Code ("Mo") -> Namen in preferences.json ("Montag")
day_names = {"Mo": "Montag", "Di": "Dienstag", "Mi": "Mittwoch", "Do": "Donnerstag", "Fr": "Freitag"}

# Popularität eines Slots berechnen - p(d, t)
def popularity(day, slot):
    weekday_pop = prefs["weekday_preferences"][day_names[day]]
    slot_pop = prefs["time_slot_preferences"][slot]["preference"]
    return weekday_pop + slot_pop

# Popularität für alle Slots berechnen
pop = {}

for v in variables:
    day, slot, _ = v.split("_")
    pop[v] = popularity(day, slot)

# Beispiel: {'Mo_A_R1': 9, 'Mo_A_R2': 9, 'Mo_B_R1': 10, ...}
#print(pop)

POP_TOL = 0.5   # erlaubte Abweichung der mittleren Popularität vom Wochenmittel

popTotal = sum(pop.values())
#print(f"pT: {popTotal}")

print("mittlere Popularität aller Slots:", popTotal / N_SLOTS)

popOverview = pd.DataFrame(
    {
        d: {
            t: popularity(d, t) 
            for t in slots
        } 
        for d in weekdays
    }
)

print("")
print("Popularität pro Tag/Slot:")

display(popOverview)

mittlere Popularität aller Slots: 6.65

Popularität pro Tag/Slot:


,Mo,Di,Mi,Do,Fr
A,9,8,8,7,5
B,10,9,9,8,6
C,7,6,6,5,3
D,7,6,6,5,3


$$\left|\sum_v (40\,p(v) - 266)\,x_{v,s}\right| \le 40\,\tau\, n_s$$

In [621]:
def add_popularity_constraints(model, x, programs):
    #popTotal = sum(pop.values())     # 266 = Summe der Popularität aller 40 Slots # schon oben definiert
    tol = round(POP_TOL * N_SLOTS)    # 0.5 * 40 = 20 (Toleranz, ebenfalls mit 40 multipliziert)

    for s in programs:
        n_s = sum(x[v, s] for v in variables)   # Anzahl Slots des Studiengangs (lineare Summe, wie beim Bedarf)

        # sum(40 * p_v - 266) * X_v,s
        deviation = sum(
            (N_SLOTS * pop[v] - popTotal) * x[v, s] 
            for v in variables
        )

        model.Add(deviation <= tol * n_s)    # nicht zu viele attraktive Slots
        model.Add(deviation >= -tol * n_s)   # nicht zu viele unattraktive Slots

# 4.4 Parallele und aufeinanderfolgende Belegung 

#### Begriffe
- **Parallel:** Ein Studiengang hat zum selben Zeitpunkt $(d,t)$ beide Räume: $x_{d,t,R1} = x_{d,t,R2} = s$.
- **Aufeinanderfolgend:** Ein Studiengang hat am selben Tag zwei direkt aufeinanderfolgende Zeitslots (A–B, B–C, C–D), egal in welchem Raum. Für die Kursgruppen zählt der zeitliche Ablauf, nicht der Raum. Den Übergang über Nacht (D → A) behandelt 4.5.

#### Regeln aus `preferences.json`
Für jeden Studiengang wird die Bedingung (`course_groups >= 3` usw.) mit seiner Gruppenzahl $g_s$ ausgewertet:

| Bedingung | Datensatz | Constraint | betrifft |
|---|---|---|---|
| $g_s = 1$ | parallel und aufeinanderfolgend `allowed: false` | beides verboten | INF (W1) |
| $g_s = 2$ | beides erlaubt, Präferenz 0 / 0 | keine Einschränkung | ET (W2, W3) |
| $g_s \ge 3$ | beides erlaubt, parallel 6 > aufeinanderfolgend 3 | (a) #parallel ≥ #aufeinanderfolgend <br> (b) #parallel $\ge \lfloor 0{,}5 \cdot bounds_{min} / 2 \rfloor$ <br> verhindert dass 0>=0 akzeptiert ewrden <br> fordert Hälfte des Min-Bedarfs des Studiengangs in dieser Woche ein | alle anderen |

#### Begründung für $g_s \ge 3$
Aus den Präferenzen (parallel 6 > aufeinanderfolgend 3 > 0) folgt, dass große Studiengänge einen wesentlichen Teil ihrer Slots parallel erhalten sollen (a). Der Datensatz nennt keinen konkreten Anteil. Wir fordern etwa die Hälfte der Slots als Parallelpaare (b) und fordern dazu:
$$\left\lfloor \frac{\text{minBedarfRäume} \cdot 0{,}5}{2} \right\rfloor$$

Ein höherer Anteil (z. B. 6/9 = 2/3) ist nicht erfüllbar: In Woche 1 lassen die Tagesgrenzen für ES (6 Slots, max. 2 pro Tag) höchstens ein Parallelpaar zu.
    - ```Hier kollidieren zwei Contraints: Tagesgrenze vs Parallelwunsch```

Erkenntnis außerdem: Ob 6/3, 10/3 oder whatever: Wichtig ist nur der größere Präferenzwert für Parallelität um diese zu bevorzugen.    

#### Umsetzung in CP-SAT
Hilfsvariablen je Tag $d$, Zeitslot $t$ und Studiengang $s$:
- `par = min(x[R1], x[R2])` ist 1, wenn $s$ beide Räume hat
    - weil wenn bei einem boolean Paar (0/1, 0/1) der kleinere Wert 1 ist, dann müssen beide 1 sein
- `used = max(x[R1], x[R2])` ist 1, wenn $s$ mindestens einen Raum hat
    - weil wenn bei einem boolean Paar (0/1, 0/1) der größere Wert 1 ist, dann ist entweder einer von beiden oder beide 1
- `seq = min(used[t], used[t+1])` ist 1, wenn $s$ zwei aufeinanderfolgende Zeitslots hat
    - analog par() bzgl. Verwendung von $min$

In [622]:
PAR_SHARE = 0.5   # Mindestanteil der Slots in Parallelpaaren (für g >= 3)

def structure_rule(g):
    # passende Regel aus preferences.json zur Gruppenzahl g finden
    # condition ist ein String wie "course_groups >= 3" und wird mit eval ausgewertet
    for rule in prefs["appointment_structure_preferences"]:
        if eval(rule["condition"], {"course_groups": g}):
            return rule

for g in [1, 2, 3, 10]:
    r = structure_rule(g)
    print(f"{g} Gruppen -> parallel: {r['parallel']}  seq: {r['sequential']}")

1 Gruppen -> parallel: {'preference': 0, 'allowed': False}  seq: {'preference': 0, 'allowed': False}
2 Gruppen -> parallel: {'preference': 0, 'allowed': True}  seq: {'preference': 0, 'allowed': True}
3 Gruppen -> parallel: {'preference': 6, 'allowed': True}  seq: {'preference': 3, 'allowed': True}
10 Gruppen -> parallel: {'preference': 6, 'allowed': True}  seq: {'preference': 3, 'allowed': True}


In [ ]:
def add_structure_constraints(model, x, week_df, programs):
    groups = dict(zip(week_df["study_program"], week_df["course_groups"] * week_df["cohorts"]))
    bounds_week = demand_bounds(week_df)

    # Methode wird im jeweiligen Wochenkontext gerufen
    
    # ... und iteriert dann über Studiengänge der Woche

    for s in programs:

        # ... extrahiert passende Struktur-Regeln
        rule = structure_rule(groups[s])
        parallel = []
        sequential = []

        # ... iteriert Tage und deren Slots

        for day in weekdays:
            used = {}
            for t in slots:

                # das hier ist der Verweis auf "x", unser Variablensystem
                # Dabei wird jeweils R1 und R2 betrachtet, sodass deren Zustand 
                # im weiteren Verlauf herangezogen werden kann.
                r1 = x[f"{day}_{t}_R1", s]
                r2 = x[f"{day}_{t}_R2", s]

                # Etablierung der Solver Variable "par"
                par = model.NewBoolVar(f"par_{day}_{t}_{s}")
                # Vergleich von Raum1 und Raum2 im aktuellen Slot
                model.AddMinEquality(par, [r1, r2])          # 1, wenn s beide Räume hat
                parallel.append(par)

                # Etablierung der Solver-Variable "used"
                used[t] = model.NewBoolVar(f"used_{day}_{t}_{s}")
                # Vergleich von Raum1 und Raum2 im aktuellen Slot
                model.AddMaxEquality(used[t], [r1, r2])      # 1, wenn s mind. einen Raum hat

            # Reihenfolgen Thema ....
            for t1, t2 in zip(slots, slots[1:]):             # (A,B), (B,C), (C,D)
                seq = model.NewBoolVar(f"seq_{day}_{t1}{t2}_{s}")
                # Rückschau auf used-Variable
                model.AddMinEquality(seq, [used[t1], used[t2]])   # 1, wenn s beide Zeitslots hat
                sequential.append(seq)

        # g = 1: allowed = false -> verboten
        if not rule["parallel"]["allowed"]:
            model.Add(sum(parallel) == 0)
        if not rule["sequential"]["allowed"]:
            model.Add(sum(sequential) == 0)

        # g >= 3: parallel bevorzugt (6 > 3)
        if rule["parallel"]["preference"] > rule["sequential"]["preference"]:
            model.Add(sum(parallel) >= sum(sequential))                                     # (a)
            model.Add(sum(parallel) >= math.floor(bounds_week[s][0] * PAR_SHARE / 2))       # (b)

In [624]:
# bereitet Lösung hinsichtlich Popularität und Struktur für Darstellung auf

def pop_and_struc(solution):
    rows = []

    # Jeden Studiengang einzeln betrachten
    for s in sorted(set(solution.values())):

        # Alle Slots dieses Studiengangs sammeln
        own = []
        for v, sg in solution.items():
            if sg == s:
                own.append(v)

        # Die belegten Tage und Zeitslots speichern
        times = set()
        for v in own:
            teile = v.split("_")
            tag = teile[0]
            slot = teile[1]
            times.add((tag, slot))

        # Gleichzeitige Prüfungen in beiden Räumen zählen
        parallel = 0
        for d in weekdays:
            for t in slots:
                if (solution[f"{d}_{t}_R1"] == s and
                    solution[f"{d}_{t}_R2"] == s):
                    parallel += 1

        # Direkt aufeinanderfolgende Prüfungen zählen
        sequential = 0
        for d in weekdays:
            for i in range(len(slots) - 1):
                t1 = slots[i]
                t2 = slots[i + 1]

                if (d, t1) in times and (d, t2) in times:
                    sequential += 1

        # Durchschnittliche Popularität berechnen
        durchschnitt = sum(pop[v] for v in own) / len(own)

        # Ergebnisse speichern
        rows.append({
            "SG": s,
            "Slots": len(own),
            "mittl. Popularität": round(durchschnitt, 2),
            "parallel": parallel,
            "aufeinanderfolgend": sequential
        })

    # Ergebnisse als Tabelle zurückgeben
    return pd.DataFrame(rows).set_index("SG")

# 4.5 Weitere Constraints

Ergänzen Sie weitere fachlich sinnvolle Bedingungen. Dazu gehören (mindestens) die folgenden:

- Ein Raum kann zu einem Zeitpunkt nur einem Studiengang zugeordnet werden.
  - erfüllt mittels ```addExactlyOne``` Constraint (siehe 4.1)

- In einer Woche dürfen nur die für diese Woche angegebenen Studiengänge Slots erhalten.
  - erfüllt mittels "Domain" (siehe 4.1)
  - im Falle der *ortools* Lib allerdings sicherzustellen durch Variablenerstellung nur für die Studiengänge der jeweiligen Woche (also keine dedizierte Bekanntgabe einer Domain wie bei der Lib aus der Vorlesung möglich)

- Ein Studiengang soll nach Möglichkeit an jedem relevanten Prüfungstag berücksichtigt werden.
  - erfüllt durch Bedarfsberücksichtigung (sieh 4.2)

Weitere:

- Sehr ungünstige Übergänge, etwa ein später Termin an einem Tag direkt gefolgt von einem frühen Termin am nächsten Tag, sollen vermieden werden (Fairness).
  - erfüllt durch `add_transition_constraints` (siehe unten): Für jeden Studiengang $s$ und jeden Tageswechsel $d \to d+1$ gilt
    $x_{(d,D,r),\,s} + x_{(d+1,A,r'),\,s} \le 1$ für alle Raumkombinationen $r, r' \in \{R1, R2\}$.
  - Das sind **binäre Constraints** zwischen je zwei Variablen: pro Studiengang 4 Tageswechsel × 4 Raumkombinationen = 16 Stück.
  - `check_transitions` prüft das zusätzlich am fertigen Plan, unabhängig vom Solver.
- Wenn ein Studiengang den letzten Slot eines Tages hat, so darf er am nächsten Tag nicht den ersten Slot haben.
  - konkrete Form der vorherigen Anforderung, umgesetzt durch denselben Constraint
- Parallele Nutzung beider Räume durch denselben Studiengang soll nur dann zulässig oder erforderlich sein, wenn dies aufgrund seiner Kursgruppenzahl sinnvoll ist.
  - erfüllt durch die Strukturregeln aus `preferences.json` (siehe 4.4): bei 1 Kursgruppe verboten, bei 2 erlaubt, ab 3 Kursgruppen eine Mindestanzahl an Parallelpaaren

(Die genaue Formulierung dieser Anforderungen als überprüfbare Constraints ist Teil der Prüfungsleistung. Achten Sie darauf, dass die Gesamtheit der Bedingungen nicht ohne fachliche Begründung so streng gewählt wird, dass regulär keine Lösung mehr existiert.)

In [ ]:
# 4.5 Kein Übergang vom letzten Slot (D) eines Tages zum ersten Slot (A) des Folgetages

def add_transition_constraints(model, x, programs):
    for s in programs:
        for day, next_day in zip(weekdays, weekdays[1:]):        # (Mo,Di), (Di,Mi), (Mi,Do), (Do,Fr)
            for r_late in rooms:
                for r_early in rooms:
                    late = x[f"{day}_D_{r_late}", s]              # s hat am Tag d den letzten Slot
                    early = x[f"{next_day}_A_{r_early}", s]       # s hat am Tag d+1 den ersten Slot
                    model.Add(late + early <= 1)                  # nicht beides
    # je Studiengang 4 Tageswechsel x 2 x 2 Raumkombinationen = 16 binäre Constraints


# Kontrolle am fertigen Plan (nach Colins Prüfung), unabhängig vom Solver
def check_transitions(solution):
    errors = []
    for s in sorted(set(solution.values())):
        for day, next_day in zip(weekdays, weekdays[1:]):
            late = any(solution[f"{day}_D_{r}"] == s for r in rooms)
            early = any(solution[f"{next_day}_A_{r}"] == s for r in rooms)
            if late and early:
                errors.append(f"{s}: letzter Slot am {day}, erster Slot am {next_day}")
    return errors

In [ ]:
# Kontrolle: enthalten die Lösungen aller Wochen noch Übergänge spät -> früh?

for week in weeks:
    model, x = build_csp(week)
    sol = solve(model, x)
    if sol is None:
        print(f"{week}: keine Lösung")
        continue
    errors = check_transitions(sol)
    print(f"{week}: {len(errors)} Übergänge spät -> früh", errors)

# 4.6 Implementierung und Lösungsbestimmung

siehe oben

# 4.7 Analyse des Lösungsraums

Untersuchen Sie für mindestens eine ausgewählte Prüfungswoche, ob mehrere zulässige Lösungen existieren. Bestimmen Sie mehrere unterschiedliche Lösungen und analysieren Sie deren Struktur.

Untersuchen Sie insbesondere:


### Welche Constraints reduzieren den Lösungsraum besonders deutlich?
- Tagesbedarf (siehe 4.2) verhindert, dass Studiengänge große Teile einzelner Tage einnehmen
- Popularität (siehe 4.3) bzw. der Fairness-Constraint engen die mittlere Popularität der Studiengänge stark ein


### Welche Auswirkungen haben die Bedarfs- und Popularitäts-Fairness-Constraints?

Einschränkung der möglichen Lösungen.

- **Bedarf:**
  - Ermittlung der Verhältnisses von Bedarfseinheiten zu Gesamtgruppenanzahl
  - Verwendung dieses Wertes für anteilige Berechnung an den 40- bzw. 8 Slots pro Woche bzw. Tag
  - Ab- bzw. Aufrunden für Ganzzahligkeit
  - Erhebung eines Constraints für Einhaltung des konkreten Bedarfes innerhalb der definierten Unter- bzw. Oberschranke
- **Popularität:**
  - Ermittlung der mittleren Popularität über die Woche hinweg für alle Studiengänge
  - Ermittlung der mittleren Popularität eines konkreten Studienganges
  - Erhebung eines Constraints der diese Popularität unter Berücksichtigung der Abweichung TAU in einem bestimmten Wertebereich hält (siehe 4.3)
  


### Welche Zuordnungen können zwischen verschiedenen Lösungen variieren?

```TODO: Lieber nur auf Lösungen +++```

Gewählte Modellierung (Variante 1):
- Anfangs war die Frage, was Variablen und was Domain sein sollen
- Zunächst wurde klar, dass das empfohlene ```ortools``` die Begriffe gar nicht explizit unterscheidet
- Modellierung der Variablen dann **entlang der Slots** (Variante 1)
- Einbettung der Studiengänge in einen Schlüssel der dann die Binärvariable auflöst
    - x[slot, studiengang]
- Absicherung dass ein Slot nur 1 Studiengang aufnehmen kann mittels ```AddExactlyOne```Constraint
- Suchraum: $5^40$

Mögliche Alternative (Variante 2):
- Möglich wäre auch gewesen entlang der **Bedarfseinheiten** der Studiengänge (z.B. "MB-1") zu modellieren
- Dann bräuchte man pro Studiengang soviele Variablen wie es Gruppen des Studiengangs gibt
- Also z.B. bei "MB" -> 16 Variablen 
- Insgesamt also wieder 40 Variablen
- Wertebereich wären dann die Slots ("Mo_A_R1", ...) => 40
- Suchraum entsprechend: $40^{40}$
- Das Netz wäre unnötig groß, da pro Zeile (Bedarfseinheit) ohnehin nur ein Slot infrage kommt, also ein Flag g- **Bedarf:**
  - Ermittlung der Verhältnisses von Bedarfseinheiten zu Gesamtgruppenanzahl
  - Verwendung dieses Wertes für anteilige Berechnung an den 40- bzw. 8 Slots pro Woche bzw. Tag
  - Ab- bzw. Aufrunden für Ganzzahligkeit
  - Erhebung eines Constraints für Einhaltung des konkreten Bedarfes innerhalb der definierten Unter- bzw. Oberschranke
- **Popularität:**
  - Ermittlung der mittleren Popularität über die Woche hinweg für alle Studiengänge
  - Ermittlung der mittleren Popularität eines konkreten Studienganges
  - Erhebung eines Constraints der diese Popularität unter Berücksichtigung der Abweichung TAU in einem bestimmten Wertebereich hält (siehe 4.3)
esetzt sein kann, alle anderen 39 "Kästchen" bleiben leer
- Man könnte es so sagen: Die Bedarfseinheiten strecken das Modell und lassen somit aber große Lücken in der Ergebnismatrix klaffen
- Unsere verwendete Variante 1 löst das besser, weil die Bedarfseinheiten sich aus der Summe der gewählten Slots ergeben und über Constraints eingefordert werden
- Constraints bei Variante 2:
    - Bedarf: wäre dann schon über Anzahl der Variablen abgedeckt (z.B. MB -> 16)
    - Doppelvergabe: Man müsste sicherstellen, dass der selbe Slot nicht mehrfach zugewiesen ist
    - Struktur: man muss nun erst durch alle Slots und Zeit und Raum extrahieren

**Variante 1** (hier im Jupyter Book verwendet!):
- Mo_A_R1, { MB = 0, ES = 0, INF = 1, ... } // len(5)
- Mo_A_R2, { MB = 1, ES = 0, INF = 0, ... }
- Mo_A_R1, { MB = 1, ES = 0, INF = 0, ... }
- ...
(40 x 5)

**Variante 2** (Alternative entlang der Bedarfseinheiten):
- MB1, { Mo_A_R1 = 0, Mo_A_R2 = 1, ... } // len(40)
- MB2, { ... }
- MB3, { ... }
- ...
(40 x 40)

```TODO +++++++++++++++++++++++```

## Umsetzung: mehrere Lösungen bestimmen

Vorgehen (Ansatz von Colin, angepasst an unser Modell mit `x[v, s]`):

1. Modell der Woche mit `build_csp` aufbauen.
2. Für jede bereits gefundene Lösung einen **Blocking-Constraint** ergänzen: Von ihren 40 Zuordnungen $x_{v,s} = 1$ dürfen höchstens 39 wieder gleich sein. Mindestens ein Slot muss also anders belegt werden.
3. Neu lösen, bis $k$ Lösungen gefunden sind oder keine weitere existiert.

CP-SAT sucht standardmäßig mit mehreren Threads parallel und liefert dadurch bei jedem Lauf andere Lösungen. `solve_reproducible` verwendet deshalb einen einzelnen Such-Thread und einen festen Seed, damit die Ergebnisse unten reproduzierbar sind.

In [ ]:
# 4.7 Mehrere Lösungen bestimmen (nach Colins Ansatz: jede gefundene Lösung per Blocking-Constraint ausschließen)

def solve_reproducible(model, x, seed=42):
    # wie solve(), aber mit einem Such-Thread und festem Seed -> gleiche Lösungen bei jedem Lauf
    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 1
    solver.parameters.random_seed = seed
    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None
    return {v: s for (v, s), var in x.items() if solver.Value(var)}

def enumerate_solutions(week, k=5):
    found = []
    for i in range(k):
        model, x = build_csp(week)

        # jede bereits gefundene Lösung ausschließen:
        # von ihren 40 Zuordnungen x[v, s] = 1 darf höchstens 39 wieder gleich sein
        for prev in found:
            model.Add(
                sum(x[v, s] for v, s in prev.items()) <= len(prev) - 1
            )

        sol = solve_reproducible(model, x)
        if sol is None:
            break
        found.append(sol)
    return found

week_47 = "Woche 1"
sols_47 = enumerate_solutions(week_47, k=5)
print(f"{len(sols_47)} unterschiedliche Lösungen für {week_47}\n")

for i, sol in enumerate(sols_47[:2]):   # zwei Pläne zum Vergleich anzeigen
    print(f"--- Lösung {i+1} ---")
    display(show_plan(sol))

### Wie stark unterscheiden sich die Lösungen?

- **Unterschiedliche Slots:** Anzahl der 40 Slots, die zwei Lösungen verschieden belegen.
- **Unterschiedliche Zeitpunkte:** wie oben, aber R1 und R2 gelten als gleichwertig. Gezählt wird, an wie vielen der 20 Zeitpunkte (Tag, Slot) die beiden Räume insgesamt anders verteilt sind. So lässt sich erkennen, ob sich Lösungen nur durch vertauschte Räume unterscheiden.

In [ ]:
# Wie stark unterscheiden sich die Lösungen?

# Anzahl Slots, die zwei Lösungen unterschiedlich belegen
def slot_diff(a, b):
    return sum(a[v] != b[v] for v in variables)

# wie slot_diff, aber R1 und R2 als gleichwertig betrachtet:
# zählt Zeitpunkte (Tag, Slot), an denen die beiden Räume insgesamt anders verteilt sind
def time_diff(a, b):
    return sum(
        sorted([a[f"{d}_{t}_R1"], a[f"{d}_{t}_R2"]]) != sorted([b[f"{d}_{t}_R1"], b[f"{d}_{t}_R2"]])
        for d in weekdays for t in slots
    )

labels = [f"L{i+1}" for i in range(len(sols_47))]
print("Unterschiedliche Slots (von 40):")
display(pd.DataFrame([[slot_diff(a, b) for b in sols_47] for a in sols_47], index=labels, columns=labels))

print("Unterschiedliche Zeitpunkte, Räume getauscht ignoriert (von 20):")
display(pd.DataFrame([[time_diff(a, b) for b in sols_47] for a in sols_47], index=labels, columns=labels))

### Welche Zuordnungen variieren, was bleibt gleich?

Die Tabelle zeigt pro Slot alle Studiengänge, die dort in den fünf Lösungen vorkommen. Darunter stehen die Kennzahlen aus `pop_and_struc` für jede Lösung nebeneinander.

In [ ]:
# Welche Zuordnungen variieren? Pro Slot alle Studiengänge, die in den Lösungen dort vorkommen

variation = {v: sorted({sol[v] for sol in sols_47}) for v in variables}
constant = [v for v, s in variation.items() if len(s) == 1]
print(f"Slots mit immer gleicher Belegung: {len(constant)} von {N_SLOTS}  {constant}")

var_plan = show_plan({v: "/".join(s) for v, s in variation.items()})
display(var_plan)

# Was bleibt gleich, was ändert sich? Kennzahlen je Lösung nebeneinander
kpis = pd.concat(
    {f"L{i+1}": pop_and_struc(sol) for i, sol in enumerate(sols_47)},
    axis=1
)
for col in ["Slots", "mittl. Popularität", "parallel", "aufeinanderfolgend"]:
    print(col)
    display(kpis.xs(col, axis=1, level=1))

### Wie groß ist der Lösungsraum?

Mit `enumerate_all_solutions` zählt CP-SAT alle Lösungen auf. Nach 10 Sekunden wird abgebrochen. Die Zahl ist also eine **Untergrenze**. Zusätzlich wird für jedes Paar (Slot, Studiengang) geprüft, ob es in irgendeiner zulässigen Lösung vorkommt.

In [ ]:
# Wie viele Lösungen gibt es? Zählen, bis das Zeitlimit erreicht ist (= Untergrenze)

class SolutionCounter(cp_model.CpSolverSolutionCallback):
    def __init__(self):
        super().__init__()
        self.count = 0
    def on_solution_callback(self):
        self.count += 1

for week in weeks:
    model, x = build_csp(week)
    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.parameters.max_time_in_seconds = 10
    counter = SolutionCounter()
    status = solver.Solve(model, counter)
    vollstaendig = status == cp_model.OPTIMAL
    print(f"{week}: {counter.count} Lösungen in 10 s gefunden"
          + ("" if vollstaendig else " (abgebrochen, es gibt mehr)"))

In [ ]:
# Kann jeder Slot grundsätzlich jeden Studiengang bekommen?
# Für jedes Paar (Slot, Studiengang): gibt es irgendeine zulässige Lösung mit x[v, s] = 1?

def possible_assignments(week):
    possible = set()
    model, x = build_csp(week)
    for (v, s), var in x.items():
        if (v, s) in possible:
            continue                        # schon in einer früheren Lösung gesehen
        m = model.Clone()
        m.Add(m.GetBoolVarFromProtoIndex(var.Index()) == 1)
        sol = solve(m, x)
        if sol is not None:
            possible |= set(sol.items())    # alle Zuordnungen dieser Lösung sind möglich
    return possible, len(x)

possible, total = possible_assignments(week_47)
print(f"{week_47}: {len(possible)} von {total} Zuordnungen (Slot, Studiengang) kommen in mindestens einer Lösung vor")

### Ergebnis der Analyse (Woche 1)

**Es gibt sehr viele zulässige Lösungen.** CP-SAT findet in 10 Sekunden über 1 000 Lösungen je Woche und bricht dann ab. Die tatsächliche Zahl ist deutlich größer. Jede der 200 Zuordnungen (Slot, Studiengang) kommt in mindestens einer Lösung vor.

**Die Lösungen unterscheiden sich stark.** Je zwei der fünf Lösungen belegen 16–32 der 40 Slots unterschiedlich. Auch wenn vertauschte Räume ignoriert werden, unterscheiden sich 12–20 der 20 Zeitpunkte. Die Vielfalt kommt also nicht nur von der Raumsymmetrie R1 ↔ R2.

**Was in allen Lösungen gleich bleibt:**
- **Slotzahl je Studiengang** (ES 6, INF 2, MB 16, MT 8, DS 8). Sie ist in Woche 1 durch den Wochenbedarf exakt festgelegt.
- **INF** ist nie parallel und nie aufeinanderfolgend. Das erzwingt schon die Tagesgrenze von höchstens 1 Slot pro Tag.
- **ES** hat immer genau 1 Parallelpaar. Die Mindestanzahl (4.4) und die Tagesgrenze von höchstens 2 Slots lassen keinen Spielraum.

**Was variiert:**
- **Lage der Slots:** welcher Tag, welche Uhrzeit, welcher Raum. Nur `Do_B_R1` und `Fr_C_R1` sind in allen fünf Lösungen gleich belegt. Das ist ein Zufall der Stichprobe, denn laut der 200-von-200-Prüfung könnten auch diese Slots anders belegt sein.
- **Mittlere Popularität:** Je Studiengang liegt sie zwischen 6,25 und 7,12, immer innerhalb des erlaubten Bandes von 6,15 bis 7,15.
- **Struktur der großen Studiengänge:** MB hat 4–5 Parallelpaare (Minimum 4) und 4–5 aufeinanderfolgende Paare, MT und DS haben je 2–3 Parallelpaare.

**Folgerung:** Die Constraints legen fest, *wie viele* Slots jeder Studiengang erhält und *wie gut* sie im Mittel sind (Anzahl, Popularitätsband, Struktur). *Wo* genau die Slots liegen, legen sie kaum fest. Alle gefundenen Lösungen sind im Sinne des CSP gleichwertig, keine ist besser als eine andere.

# 4.8 Experiment: Veränderung eines Constraints

Verändern Sie für eine ausgewählte Prüfungswoche gezielt eine Randbedingung, zum Beispiel:
- einen Popularitätswert eines Tages oder Zeitslots,
- die Zahl der Kursgruppen eines Studiengangs,
- die Menge der in einer Woche beteiligten Studiengänge,
- die Toleranz eines Fairness-Constraints oder
- eine Regel zur parallelen bzw. aufeinanderfolgenden Belegung.

Formulieren Sie vor der Berechnung eine begründete Hypothese über die erwarteten Auswirkungen auf Bedarfsverteilung, zulässige Zuordnungen und Lösungsraum. Führen Sie anschließend die Berechnung (ggf. mehrfach) erneut durch und vergleichen Sie Erwartung und Ergebnis.

```TODO +++++++++++++++++++++++```

## Gewählte Änderung: Toleranz τ der Popularitäts-Fairness (Woche 1)

Wir variieren $\tau$ (`POP_TOL`) schrittweise von 2,0 bis 0,1. Alle anderen Constraints bleiben unverändert. Erlaubt ist für jeden Studiengang eine mittlere Popularität im Band $\bar p \pm \tau = 6{,}65 \pm \tau$ (siehe 4.3).

### Hypothese (vor der Berechnung)

**Bedarfsverteilung:** bleibt unverändert. Die Slotzahlen hängen nur vom Wochen- und Tagesbedarf ab, nicht von der Popularität. In Woche 1 sind sie mit 6/2/16/8/8 exakt festgelegt.

**Zulässige Zuordnungen:** Je kleiner $\tau$, desto weniger Kombinationen guter und schlechter Slots sind für einen Studiengang erlaubt. Studiengänge mit wenigen Slots trifft das zuerst. Ihr Mittelwert kann nur grob gestuft liegen:
- **INF** (2 Slots): Der Mittelwert liegt immer auf einem Vielfachen von 0,5 (… 6,0 · 6,5 · 7,0 …).
- Der Wert 7,0 ist nur zulässig, solange $6{,}65 + \tau \ge 7{,}0$ gilt, also für $\tau \ge 0{,}35$.
- Der Wert 6,5 ist nur zulässig, solange $6{,}65 - \tau \le 6{,}5$ gilt, also für $\tau \ge 0{,}15$.

**Lösungsraum:** Er schrumpft mit kleinerem $\tau$. Für $\tau < 0{,}15$ erwarten wir **Unlösbarkeit**, weil INF dann keinen zulässigen Mittelwert mehr erreichen kann. Die Studiengänge mit vielen Slots sind feiner gestuft und begrenzen nicht: MB mit 16 Slots in Schritten von 0,0625, ES mit 6 Slots in Schritten von 0,167. Bei großem $\tau$ (2,0) ist der Fairness-Constraint praktisch wirkungslos. Die mittlere Popularität einzelner Studiengänge darf dann stark streuen.

### Berechnung

Je $\tau$ werden ermittelt:
- **lösbar:** ob es überhaupt eine Lösung gibt.
- **Slots je Studiengang:** in der Reihenfolge ES/INF/MB/MT/DS.
- **Ø Popularität möglich:** die kleinste und größte erreichbare mittlere Popularität von INF und MB. Dafür wird der Solver einmal minimierend und einmal maximierend aufgerufen. Das dient nur der Analyse, das Modell selbst optimiert nichts.
- **Lösungen in 5 s:** wie viele Lösungen in 5 Sekunden aufgezählt werden. Das ist eine Untergrenze und nur ein Richtwert für die Größe des Lösungsraums.

`POP_TOL` wird danach wieder auf den Originalwert 0,5 gesetzt. Die Zelle läuft etwa eine Minute.

In [ ]:
# 4.8 Experiment: Toleranz τ der Popularitäts-Fairness variieren (Woche 1)

def count_solutions(model, seconds=5):
    # Lösungen aufzählen, bis das Zeitlimit erreicht ist (Untergrenze, nur als Richtwert)
    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.parameters.max_time_in_seconds = seconds
    counter = SolutionCounter()                      # aus 4.7
    status = solver.Solve(model, counter)
    return counter.count, status == cp_model.OPTIMAL

def popularity_range(week, s):
    # kleinste und größte erreichbare mittlere Popularität von Studiengang s
    result = []
    for sense in ("min", "max"):
        model, x = build_csp(week)
        total = sum(pop[v] * x[v, s] for v in variables)
        model.Minimize(total) if sense == "min" else model.Maximize(total)
        sol = solve_reproducible(model, x)
        own = [v for v, sg in sol.items() if sg == s]
        result.append(sum(pop[v] for v in own) / len(own))
    return result

week_48 = "Woche 1"
mean_pop = popTotal / N_SLOTS
tau_original = POP_TOL
rows = []

try:
    for tau in [2.0, 1.0, 0.5, 0.3, 0.2, 0.15, 0.1]:
        POP_TOL = tau                                # wird in add_popularity_constraints gelesen
        sol = solve_reproducible(*build_csp(week_48))
        row = {
            "τ": tau,
            "erlaubtes Band": f"{mean_pop - tau:.2f} – {mean_pop + tau:.2f}",
            "lösbar": sol is not None,
        }
        if sol is not None:
            kpi = pop_and_struc(sol)
            n, complete = count_solutions(build_csp(week_48)[0])
            inf_min, inf_max = popularity_range(week_48, "INF")
            mb_min, mb_max = popularity_range(week_48, "MB")
            row.update({
                "Slots je SG": "/".join(str(kpi.loc[s, "Slots"]) for s in ["ES", "INF", "MB", "MT", "DS"]),
                "Ø Pop. INF möglich": f"{inf_min:.2f} – {inf_max:.2f}",
                "Ø Pop. MB möglich": f"{mb_min:.2f} – {mb_max:.2f}",
                "Lösungen in 5 s": f"{n}" + ("" if complete else "+"),
            })
        rows.append(row)
finally:
    POP_TOL = tau_original                           # Originalwert für alle folgenden Zellen

display(pd.DataFrame(rows).set_index("τ").fillna("–"))

### Vergleich: Erwartung und Ergebnis

| Hypothese | Ergebnis | bestätigt? |
|---|---|---|
| Slotzahlen bleiben gleich | bei jedem lösbaren $\tau$: 6/2/16/8/8 | ja |
| INF verliert als erster Studiengang Möglichkeiten | Bereich von INF: 5,00–8,50 bei $\tau = 2{,}0$ → 6,50–7,00 bei 0,5 → nur noch 6,50 ab 0,3 | ja, 7,0 fällt wie vorhergesagt unter $\tau = 0{,}35$ weg |
| große Studiengänge werden fein gestuft eingeengt | Bereich von MB: 5,25–8,12 → 6,19–7,12 → 6,56–6,75 bei 0,15 | ja |
| Lösungsraum schrumpft | Lösungen in 5 s: mehrere Tausend bei 2,0 → wenige Hundert ab 0,3 | ja, als Tendenz (Zählwerte sind zeitabhängig und schwanken leicht) |
| unlösbar für $\tau < 0{,}15$ wegen INF | lösbar bei 0,15, unlösbar bei 0,1 | ja, Grenze genau vorhergesagt |

**Nicht erwartet:** Auch bei $\tau = 2{,}0$ schöpft INF das erlaubte Band (4,65–8,65) nicht aus, sondern erreicht nur 5,00–8,50. Die übrigen Constraints begrenzen die extremen Kombinationen also zusätzlich, auch ohne wirksamen Fairness-Constraint.

**Fazit:** Die Toleranz wirkt nur auf die Popularität, nicht auf die Bedarfsverteilung. Wie lösbar das Modell ist, entscheidet der Studiengang mit den wenigsten Slots. Bei INF sind nur Mittelwerte in Schritten von 0,5 möglich, deshalb liegt die kritische Grenze bei $\tau = 6{,}65 - 6{,}5 = 0{,}15$. Unser gewähltes $\tau = 0{,}5$ liegt deutlich darüber und lässt INF noch zwei mögliche Mittelwerte (6,5 und 7,0).

# 4.9 -  Unlösbarkeit und Constraint-Konflikte

### Alle parallel

-> Kann nicht funktionieren


### Warum kann Woche 1 bei `τ = 0.1` unlösbar sein?

Wenn ein Studiengang nur zwei Slots bekommt, berechnet sich sein Durchschnitt so:

$$
\bar p_s = \frac{p_1 + p_2}{2}
$$

Da die Popularitätswerte ganzzahlig sind, kann der Durchschnitt nur in Schritten von `0.5` auftreten, zum Beispiel:

$$
6.0,\quad 6.5,\quad 7.0,\quad 7.5
$$

Bei einem globalen Durchschnitt von `6.65` und einer Toleranz von `τ = 0.1` muss der Durchschnitt des Studiengangs im folgenden Bereich liegen:

$$
6.65 - 0.1 \leq \bar p_s \leq 6.65 + 0.1
$$

also:

$$
6.55 \leq \bar p_s \leq 6.75
$$

Keiner der möglichen Durchschnittswerte `6.0`, `6.5` oder `7.0` liegt in diesem Bereich. Die Bedingung kann für einen Studiengang mit zwei Slots daher nicht erfüllt werden.

### Warum hilft eine größere Toleranz?

Bei `τ = 0.5` erweitert sich der zulässige Bereich:

$$
6.65 - 0.5 \leq \bar p_s \leq 6.65 + 0.5
$$

also:

$$
6.15 \leq \bar p_s \leq 7.15
$$

Nun sind beispielsweise die Durchschnittswerte `6.5` und `7.0` zulässig.

**Fazit:** Eine kleine Toleranz kann dazu führen, dass kein möglicher Durchschnittswert die Fairnessbedingung erfüllt. Eine größere Toleranz gibt dem Solver mehr Spielraum. Das garantiert allerdings nicht automatisch eine Lösung, da auch die übrigen Nebenbedingungen erfüllt werden müssen.


```TODO +++++++++++++++++++++++```

# 4.10 Kritische Reflexion

Beantworten Sie abschließend kurz und präzise:

### Welche Modellierungsentscheidung hatte den größten Einfluss auf Ihr CSP?

CSP pro Woche
Toleranz und Popularität



### Wie haben Sie “faire Verteilung” bezüglich Bedarf und Popularität operationalisiert?

Kurz: ```Constraints mit Intervallgrenze```

- **Bedarf:**
  - Ermittlung der Verhältnisses von Bedarfseinheiten zu Gesamtgruppenanzahl
  - Verwendung dieses Wertes für anteilige Berechnung an den 40- bzw. 8 Slots pro Woche bzw. Tag
  - Ab- bzw. Aufrunden für Ganzzahligkeit
  - Erhebung eines Constraints für Einhaltung des konkreten Bedarfes innerhalb der definierten Unter- bzw. Oberschranke
- **Popularität:**
  - Ermittlung der mittleren Popularität über die Woche hinweg für alle Studiengänge
  - Ermittlung der mittleren Popularität eines konkreten Studienganges
  - Erhebung eines Constraints der diese Popularität unter Berücksichtigung der Abweichung TAU in einem bestimmten Wertebereich hält (siehe 4.3)

siehe auch 4.7 ("Welche Auswirkungen haben die Bedarfs- und Popularitäts-Fairness-Constraints?)



### Welche Auswirkungen hat die Wahl der Toleranzen auf die Lösbarkeit?

Kurz: ```Zu geringe Toleranz kann zu Unlösbarkeit führen aufgrund zu kleinteiliger Constraints```

- 0,1 -> Unlösbar für INF in Woche 1
- Grund liegt in 0.5-Schritten im Ergebnis
- 6,65 ist in W1 die mittlere Popularität
- Bei TAU=0,1 ist Untergrenze 6,55 und Obergrenze 6,75
- Somit kann keine Lösung gefunden werden


### Welche Constraints schränken den Lösungsraum besonders stark ein?

Kurz: ```Tagesbedarf und Popularität```

siehe 4.7


### Gibt es redundante Constraints?

Kurz: ```Redundanzen bei Par/Seq```

- Parallel/Sequentell Verbot dann unnötig, wenn ohnehin nur 1 Slot pro Tag
- Außerdem: Bei 2 Slots pro Tag kann nur entweder parallel oder sequentiell sein


### Welche Grenzen besitzt das Modell gegenüber einer realen Prüfungsplanung?

Kurz: ```Modell nicht schlecht, aber noch nicht vollständig hinsichtlich Details```

- Studentenanzahl, Kapazität
- Räume werden als gleich gut behandelt -> Fachliche Zuordnung (PC-Räume etc.)
- Keine Berücksichtigung von Prüfpersonen
- Bedarf anhand der Kursgruppenzahl, keine Realangabe des Bedarfs
- Regeln gelten nur pro Woche
- Je nach Constraint kann es auch gar keine Lösungen geben, reale Planung würde versuchen immer Lösung zu finden



# 5 Nutzung generativer KI

| Fragestellung | Vorschlag der GenAI | Eigene Bewertung / Entscheidung |

```TODO +++++++++++++++++++++++```